In [1]:
import torch
import torchquad

torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")
torchquad.set_up_backend("torch", "float64", True)
torchquad.set_log_level("TRACE")

2026-06-16 20:39:55.546 | INFO     | torchquad.utils.enable_cuda:enable_cuda:17 - PyTorch VERSION: 2.12.0+cu130
2026-06-16 20:39:55.573 | INFO     | torchquad.utils.enable_cuda:enable_cuda:18 - CUDNN VERSION: 92000
2026-06-16 20:39:55.573 | INFO     | torchquad.utils.enable_cuda:enable_cuda:19 - Number of CUDA Devices: 1
2026-06-16 20:39:55.574 | INFO     | torchquad.utils.enable_cuda:enable_cuda:20 - Active CUDA Device: GPU0
2026-06-16 20:39:55.574 | INFO     | torchquad.utils.set_precision:set_precision:54 - Setting Torch's default dtype to float64 and device to CUDA.
20:39:55|TQ-DEBUG| Setting LogLevel to TRACE


In [2]:
# pyright: reportOptionalMemberAccess=none, reportOptionalSubscript=none
class Batch1DIntegrator(torchquad.Gaussian):
    """Custom integrator for batch 1D integration with variable domains.

    This integrator can compute multiple integrals with different domains
    in a single call, providing significant speedup over sequential computation.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.disable_integration_domain_check = True

    def _resize_roots(self, integration_domain, roots):
        """Resize roots for batched integration domains.

        Args:
            integration_domain: Shape [batch_size, 2] for multiple domains
            roots: Shape [N] - the Gaussian quadrature nodes

        Returns:
            Resized roots with shape [batch_size, N]
        """
        if integration_domain.ndim == 1:
            # Single domain case - use parent implementation
            return super()._resize_roots(integration_domain, roots)

        # Batch case
        assert roots.ndim == 1
        assert integration_domain.ndim == 2
        assert integration_domain.shape[-1] == 2

        roots = roots.to(integration_domain.device)

        # Extract bounds for all domains
        a = integration_domain[:, 0:1]  # Shape [batch_size, 1]
        b = integration_domain[:, 1:2]  # Shape [batch_size, 1]

        # Broadcast and transform roots for each domain
        roots_expanded = roots.unsqueeze(0)  # [1, N]

        # Transform from [-1, 1] to [a, b] for each domain
        out = ((b - a) / 2) * roots_expanded + ((a + b) / 2)  # [batch_size, N]

        return out

    def integrate(self, fn, dim, N, integration_domain=None, backend="torch"):
        """Integrate function over multiple domains in a single call.

        Args:
            fn: Function to integrate
            dim: Must be 1 for this implementation
            N: Number of quadrature points
            integration_domain: Shape [batch_size, 2] for batch integration
            backend: Must be "torch"

        Returns:
            Tensor of shape [batch_size] with integral results
        """
        assert dim == 1
        assert backend == "torch"

        if integration_domain.ndim == 1:
            integration_domain = integration_domain.reshape(1, 2)

        batch_size = integration_domain.shape[0]

        # Get Gaussian quadrature points and weights
        N = self._adjust_N(dim=1, N=N)
        roots = self._roots(N, backend, integration_domain.requires_grad)
        weights = self._weights(N, dim, backend)

        # Resize roots for all domains at once
        grid_points = self._resize_roots(integration_domain, roots)  # [batch_size, N]

        # Evaluate integrand at all points
        # Flatten for function evaluation: [batch_size * N, 1]
        points_flat = grid_points.reshape(-1, 1)
        function_values = fn(points_flat)  # [batch_size * N]

        # Reshape back to [batch_size, N]
        function_values = function_values.reshape(batch_size, N)

        # Apply weights and sum for each domain
        weighted_values = function_values * weights.unsqueeze(0)

        # Scale by domain width and sum
        domain_widths = (integration_domain[:, 1] - integration_domain[:, 0]) / 2
        results = domain_widths * weighted_values.sum(dim=1)

        return results

In [3]:
Batch1DIntegrator().integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x, 1, 2, torch.tensor([8.0, 30.0]))

/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py:122: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


tensor([11058.4408], device='cuda:0')

In [4]:
import collections.abc
import typing

class Batch1DIntegrator(torchquad.Gaussian):
	"""Custom integrator for batch 1D integration with variable domains.

	This integrator can compute multiple integrals with different domains
	in a single call, providing significant speedup over sequential computation.
	"""
	backend: typing.Final = "torch"

	def __init__(self, *args, **kwargs):
		super().__init__(*args, **kwargs)
		self.disable_integration_domain_check = True

	def _resize_roots(self, integration_domain: torch.Tensor, roots: torch.Tensor):
		r"""Resize roots for batched integration domains.

		Parameters
		----------
		integration_domain: torch.Tensor, shape of (batch_size, 2)
			Integration domains for each batch element
		roots: torch.Tensor, shape of (N,)
			The Gaussian quadrature nodes

		Returns
		-------
		torch.Tensor, shape of (batch_size, N)
			Resized roots
		"""
		# Batch case
		assert roots.ndim == 1 
		assert integration_domain.ndim == 2 
		assert integration_domain.shape[-1] == 2

		# Extract bounds for all domains
		a: typing.Final[torch.Tensor] = integration_domain[:, :1] # batch_size, 1
		b: typing.Final[torch.Tensor] = integration_domain[:, 1:] # batch_size, 1

		# Transform from [-1, 1] to [a, b] for each domain
		return (b - a) / 2 * roots.to(integration_domain.device).reshape(1, -1) + (a + b) / 2

	def integrate(
		self,
		fn: collections.abc.Callable[[torch.Tensor], torch.Tensor],
		dim: int = 1,
		N: int = 8, # copied from parent default
		integration_domain: torch.Tensor = torch.tensor([-1.0, 1.0]),
		backend: typing.Any = None
	):
		r"""Integrate function over multiple domains in a single call.

		Parameters
		----------
		fn : collections.abc.Callable[[torch.Tensor], torch.Tensor]
			Function to integrate, it takes any tensor shape and return the same shape
		dim : int
			Dimensionality of the integration space, in this case must be 1
		N : int
			Number of quadrature points
		integration_domain : torch.Tensor, shape of (batch_size, 2)
			Integration domains for each batch element
		backend : Any, optional
			Integration backend and unused (using "torch" instead), by default None

		Returns
		-------
		torch.Tensor, shape of (batch_size,)
			Integral results

		Notes
		-----
		For Gaussian quadrature, the integral is approximated as:
		.. math::
			\int_a^b f(x) dx \approx \frac{b - a}{2} \sum_{i=1}^N w_i f\left(\frac{b - a}{2} r_i + \frac{a + b}{2}\right)

		where :math:`r_i` are the roots and :math:`w_i` are the weights.
		"""
		assert dim == 1
		if integration_domain.ndim == 1:
			integration_domain = integration_domain.reshape(1, -1)
		assert integration_domain.ndim == 2 and integration_domain.shape[-1] == 2
		return (integration_domain[:, 1] - integration_domain[:, 0]) / 2 * (self._weights(N, 1, Batch1DIntegrator.backend).reshape(1, -1) * fn(self._resize_roots(integration_domain, self._roots(N, Batch1DIntegrator.backend, integration_domain.requires_grad)))).sum(dim=1)


integrator: typing.Final = Batch1DIntegrator()
display(integrator.integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x, 1, 2, torch.tensor([[8.0, 30.0]] * 50))[0].item())
%timeit integrator.integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x, 1, 2, torch.tensor([[8.0, 30.0]] * 50))

11058.440781141358

736 μs ± 84.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
import typing

class Batch1DIntegrator2(torchquad.Gaussian):
	"""Custom integrator for batch 1D integration with variable domains.

	This integrator can compute multiple integrals with different domains
	in a single call, providing significant speedup over sequential computation.
	"""
	backend: typing.Final = "torch"

	def __init__(self, N: int, requires_grad: bool = False):
		super().__init__()
		self.weights = self._weights(N, 1, Batch1DIntegrator.backend)
		self.roots = self._roots(N, Batch1DIntegrator.backend, requires_grad)

	@torch.compile(fullgraph=True, backend="cudagraphs")
	def _resize_roots(self, integration_domain: torch.Tensor):
		r"""Resize roots for batched integration domains.

		Parameters
		----------
		integration_domain: torch.Tensor, shape of (batch_size, 2)
			Integration domains for each batch element

		Returns
		-------
		torch.Tensor, shape of (batch_size, N)
			Resized roots
		"""
		# Extract bounds for all domains
		a: typing.Final[torch.Tensor] = integration_domain[:, :1] # batch_size, 1
		b: typing.Final[torch.Tensor] = integration_domain[:, 1:] # batch_size, 1
		# Transform from [-1, 1] to [a, b] for each domain
		return (b - a) / 2 * self.roots + (a + b) / 2

	@torch.compile(fullgraph=True, backend="cudagraphs")
	def integrate(
		self,
		fn: collections.abc.Callable[[torch.Tensor], torch.Tensor],
		integration_domain: torch.Tensor = torch.tensor([-1.0, 1.0])
	):
		r"""Integrate function over multiple domains in a single call.

		Parameters
		----------
		fn : collections.abc.Callable[[torch.Tensor], torch.Tensor]
			Function to integrate, it takes any tensor shape and return the same shape
		dim : int
			Dimensionality of the integration space, in this case must be 1
		N : int
			Number of quadrature points
		integration_domain : torch.Tensor, shape of (batch_size, 2)
			Integration domains for each batch element
		backend : Any, optional
			Integration backend and unused (using "torch" instead), by default None

		Returns
		-------
		torch.Tensor, shape of (batch_size,)
			Integral results

		Notes
		-----
		For Gaussian quadrature, the integral is approximated as:
		.. math::
			\int_a^b f(x) dx \approx \frac{b - a}{2} \sum_{i=1}^N w_i f\left(\frac{b - a}{2} r_i + \frac{a + b}{2}\right)

		where :math:`r_i` are the roots and :math:`w_i` are the weights.
		"""
		if integration_domain.ndim == 1:
			integration_domain = integration_domain.reshape(1, -1)
		assert integration_domain.ndim == 2 and integration_domain.shape[-1] == 2
		return (integration_domain[:, 1] - integration_domain[:, 0]) / 2 * (self.weights * fn(self._resize_roots(integration_domain))).sum(dim=1)


integrator2: typing.Final = Batch1DIntegrator2(2)
display(integrator2.integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x, torch.tensor([[8.0, 30.0]] * 50))[0].item())
%timeit integrator2.integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x, torch.tensor([[8.0, 30.0]] * 50))

/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py:122: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


11058.440781141358

274 μs ± 26.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [6]:
import typing

class Batch1DIntegrator3(torchquad.Gaussian):
	"""Custom integrator for batch 1D integration with variable domains.

	This integrator can compute multiple integrals with different domains
	in a single call, providing significant speedup over sequential computation.
	"""
	backend: typing.Final = "torch"

	def __init__(
		self,
		N: int,
		integration_domain: torch.Tensor = torch.tensor([-1.0, 1.0]),
		requires_grad: bool = False
	):
		super().__init__()
		self.weights = self._weights(N, 1, Batch1DIntegrator.backend)
		self.roots = self._roots(N, Batch1DIntegrator.backend, requires_grad)
		assert integration_domain.ndim == 1 and integration_domain.shape[-1] == 2
		a: typing.Final[float] = integration_domain[0].item()
		b: typing.Final[float] = integration_domain[1].item()
		self.resized_roots = (b - a) / 2 * self.roots + (a + b) / 2
		self.factor = (b - a) / 2

	@torch.compile(fullgraph=True, backend="cudagraphs")
	def _resize_roots(self, integration_domain: torch.Tensor):
		r"""Resize roots for batched integration domains.

		Parameters
		----------
		integration_domain: torch.Tensor, shape of (batch_size, 2)
			Integration domains for each batch element

		Returns
		-------
		torch.Tensor, shape of (batch_size, N)
			Resized roots
		"""
		# Extract bounds for all domains
		a: typing.Final[torch.Tensor] = integration_domain[:, :1] # batch_size, 1
		b: typing.Final[torch.Tensor] = integration_domain[:, 1:] # batch_size, 1
		# Transform from [-1, 1] to [a, b] for each domain
		return (b - a) / 2 * self.roots.to(integration_domain.device).reshape(1, -1) + (a + b) / 2

	@torch.compile(fullgraph=True, backend="cudagraphs")
	def integrate(
		self,
		fn: collections.abc.Callable[[torch.Tensor], torch.Tensor]
	):
		r"""Integrate function over multiple domains in a single call.

		Parameters
		----------
		fn : collections.abc.Callable[[torch.Tensor], torch.Tensor]
			Function to integrate, it takes any tensor shape and return the same shape
		dim : int
			Dimensionality of the integration space, in this case must be 1
		N : int
			Number of quadrature points
		integration_domain : torch.Tensor, shape of (batch_size, 2)
			Integration domains for each batch element
		backend : Any, optional
			Integration backend and unused (using "torch" instead), by default None

		Returns
		-------
		torch.Tensor, shape of (batch_size,)
			Integral results

		Notes
		-----
		For Gaussian quadrature, the integral is approximated as:
		.. math::
			\int_a^b f(x) dx \approx \frac{b - a}{2} \sum_{i=1}^N w_i f\left(\frac{b - a}{2} r_i + \frac{a + b}{2}\right)

		where :math:`r_i` are the roots and :math:`w_i` are the weights.
		"""
		return self.factor * (self.weights * fn(self.resized_roots)).sum()


integrator3: typing.Final = Batch1DIntegrator3(2, torch.tensor([8.0, 30.0]))
display(integrator3.integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x))
%timeit integrator3.integrate(lambda x: 2000 * torch.log(140000 / (140000 - 2100 * x)) - 9.8 * x)

/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py:122: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


tensor(11058.4408, device='cuda:0')

100 μs ± 8.57 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
import typing
import math
import sympy as sp

@typing.overload
def wendland(r: torch.Tensor, half_d: int) -> torch.Tensor: ...

@typing.overload
def wendland(r: float, half_d: int) -> float: ...

def wendland(r: torch.Tensor | float, half_d: int) -> torch.Tensor | float:
	l: typing.Final[int] = half_d + 3
	return (1-r) ** l * (l * r + 1)


class Analytic0:
	@typing.final
	class __LazyList[T]:
		__slots__: typing.Final[tuple] = ("__items", "__builder")
		__items: list[T]
		__builder: collections.abc.Callable[[int], T]

		def __init__(self, builder: collections.abc.Callable[[int], T], initial: list[T] = []) -> None:
			self.__items = initial
			self.__builder = builder

		def __getitem__(self, index: int) -> T:
			if len(self.__items) <= index:
				for i in range(len(self.__items), index + 1):
					self.__items.append(self.__builder(i))
			return self.__items[index]


	@typing.final
	class __classproperty[T]:
		__slots__: typing.Final[tuple] = ("__func",)
		__func: collections.abc.Callable[..., T]

		def __init__(self, func: collections.abc.Callable[..., T]):
			self.__func = func

		def __get__(self, obj: object, cls: type | None = None) -> T:
			return self.__func(cls or type(obj))

	@staticmethod
	def __next_builder(i: int) -> list[int]:
		if i == 0:
			return [1]
		else:
			prev = Analytic0.B_kn[i-1]
			new = [i * prev[-1]]
			for j in range(i-2, -1, -1):
				new.append(new[-1] + (j+1)*prev[j])
			new.append(new[-1])
			new.reverse()
			return new

	__B_kn_table: typing.Final[__LazyList[list[int]]] = __LazyList(__next_builder)
	r_c: typing.Final[sp.Symbol] = sp.Symbol("r_c", real=True)

	@__classproperty
	def B_kn(cls) -> __LazyList[list[int]]:
		return cls.__B_kn_table

	def __init__(self, dim: int, k: int = 1) -> None:
		m = dim // 2
		l: typing.Final[int] = m + k + 1
		degree = l + 2 * k
		degree_range = range(degree + 1)
		denom: typing.Final[sp.Integer] = typing.cast(sp.Integer, sp.factorial2(2*k-1))
		# sum_{n=0}^{l+2k} alpha_{l,k,j} * r^j
		alpha_sp = [typing.cast(sp.Rational, sp.Mul(sp.Add(*(sp.Mul((-1) ** n, sp.binomial(t, n), Analytic0.B_kn[k][n]) for n in range(min(k, t) + 1))), (-1) ** t, sp.Rational(sp.binomial(degree, t), denom)).doit()) for t in degree_range]
		# sum_{n=0}^{j} alpha_{l,k,n} * alpha_{l,k,j-n}
		alpha2_sp= [typing.cast(sp.Rational, sp.Add(*(sp.Mul(alpha_sp[n], alpha_sp[j-n]) for n in range(max(0, j-degree),min(j, degree) + 1))).doit().simplify()) for j in range(0, degree * 2 + 1)]
		self.integral2 = sp.Mul(sp.pi ** m, sp.Rational(1, sp.factorial(m)), sp.Add(*(sp.Mul(alpha2_sp[j], Analytic0.r_c**(-j), sp.lowergamma(sp.Rational(j + dim, sp.Integer(2)), Analytic0.r_c**2)) for j in range(degree * 2 + 1)))).simplify()

	def __call__(self, r_c: float) -> float:
		return float(self.integral2.evalf(subs={Analytic0.r_c: r_c}))


class Batch2DIntegrator:
	"""Custom integrator for batch 2D integration

	This integrator can compute multiple integrals with different domains
	in a single call, providing significant speedup over sequential computation.
	"""
	backend: typing.Final = "torch"
	integrator: typing.Final = torchquad.Gaussian()

	def function_to_integrate(self, w: torch.Tensor, v: torch.Tensor, r_c: float, uij: torch.Tensor):
		x = torch.sqrt((w - uij / 2) ** 2 + v ** 2)
		y = torch.sqrt((w + uij / 2) ** 2 + v ** 2)
		return torch.exp(-r_c**2 * (w**2 + v**2)) * wendland(x, self.half_d) * wendland(y, self.half_d) * v ** (self.dim - 2)

	def __init__(
		self,
		dim: int
	):
		self.dim = dim
		self.half_d = dim // 2
		self.factor = 2 * (2 * math.pi) ** (self.half_d - 1) / int(sp.Integer(sp.factorial2(self.dim - 3))) # include parts of the prefactor

	@torch.compile(fullgraph=True)
	def _resize_roots(self, integration_domain: torch.Tensor, roots: torch.Tensor):
		r"""Resize roots for batched integration domains.

		Parameters
		----------
		integration_domain: torch.Tensor, shape of (..., 2)
			Integration domains for each batch element

		Returns
		-------
		torch.Tensor, shape of (..., N_in)
			Resized roots
		"""
		# Extract bounds for all domains
		a: typing.Final[torch.Tensor] = integration_domain[..., :1] # batch_size, 1
		b: typing.Final[torch.Tensor] = integration_domain[..., 1:] # batch_size, 1
		# Transform from [-1, 1] to [a, b] for each domain
		return (b - a) / 2 * roots + (a + b) / 2

	def __call__(
		self,
		N_out: int,
		N_in: int,
		r_c: float,
		u: torch.Tensor
	):
		r"""Integrate function over multiple domains in a single call.

		Parameters
		----------
		r_c : float
			Parameter for the function to integrate
		u : torch.Tensor, shape of (N,)
			Pairwise distances between 0 and 2

		Returns
		-------
		torch.Tensor, shape of (N,)
			Integral results

		Notes
		-----
		For Gaussian quadrature, the integral is approximated as:
		.. math::
			\int_a^b f(x) dx \approx \frac{b - a}{2} \sum_{i=1}^N w_i f\left(\frac{b - a}{2} r_i + \frac{a + b}{2}\right)

		where :math:`r_i` are the roots and :math:`w_i` are the weights.
		"""
		# assert torch.all(u > 0).item() and torch.all(u < 2).item()
		weights_out = Batch2DIntegrator.integrator._weights(N_out, 1, Batch2DIntegrator.backend)
		roots_out = Batch2DIntegrator.integrator._roots(N_out, Batch2DIntegrator.backend, False)
		weights_in = Batch2DIntegrator.integrator._weights(N_in, 1, Batch2DIntegrator.backend)
		roots_in = Batch2DIntegrator.integrator._roots(N_in, Batch2DIntegrator.backend, False)
		half_u: typing.Final[torch.Tensor] = u / 2
		domain_out: typing.Final[torch.Tensor] = torch.stack([torch.zeros_like(u), torch.sqrt(1-half_u**2)], dim=-1) # shape (N, 2)
		roots_out_resized: typing.Final[torch.Tensor] = self._resize_roots(domain_out, roots_out) # shape (N, N_out)
		domain_in: typing.Final[torch.Tensor] = torch.stack([half_u.reshape(-1, 1) - torch.sqrt(1.0 - roots_out_resized**2), torch.sqrt(1.0 - roots_out_resized**2) - half_u.reshape(-1, 1)], dim=-1) # shape (N, N_out, 2)
		return self.factor * r_c ** self.dim * torch.exp(-r_c**2 * half_u**2) * (domain_out[..., 1] - domain_out[..., 0]) / 2 * (weights_out * (domain_in[..., 1] - domain_in[..., 0]) / 2 * (weights_in * self.function_to_integrate(roots_out_resized.unsqueeze(-1), self._resize_roots(domain_in, roots_in), r_c, u.reshape(-1, 1, 1))).sum(-1)).sum(-1)


eps = math.sqrt(torch.finfo(torch.float).eps * torch.finfo(torch.double).eps)
print(f"eps={eps}")
b2int = Batch2DIntegrator(2)
def test_torchquad(r_c: float, pts: torch.Tensor) -> torch.Tensor:
	pdist = pts
	N_out = 32
	N_in = 32
	last = b2int(N_out, N_in, r_c, pdist)
	while pdist.numel() > 0:
		N_in *= 2
		now = b2int(N_out, N_in, r_c, pdist)
		not_conv = torch.abs(now - last) > eps
		last = now[not_conv]
		pdist = pdist[not_conv]
		# print(N_in, pdist.numel())
	pdist = pts
	idx = torch.arange(pdist.numel())
	result = torch.zeros(pdist.numel(), dtype=torch.float64)
	last = b2int(N_out, N_in, r_c, pdist)
	while pdist.numel() > 0:
		N_out *= 2
		now = b2int(N_out, N_in, r_c, pdist)
		not_conv = torch.abs(now - last) > eps
		result[idx[torch.logical_not(not_conv)]] = now[torch.logical_not(not_conv)]
		last = now[not_conv]
		pdist = pdist[not_conv]
		idx = idx[not_conv]
		# print(N_out, pdist.numel())
	return result


import warnings

with warnings.catch_warnings():
	warnings.filterwarnings("ignore", category=UserWarning)
	r_c = 2.5
	x_test = torch.arange(201) / 100.
	result = test_torchquad(r_c, x_test)
	print(r_c, result) # grid test
	N = 64
	x_grid = torch.cos(torch.arange(N + 1) * math.pi / N) + 1
	f_grid = test_torchquad(r_c, x_grid)
	w_grid = (-1.)**torch.arange(N + 1)
	w_grid[0] /= 2
	w_grid[-1] /= 2
	fit = torch.zeros(x_test.numel(), dtype=torch.float64)
	grid_dist = x_test.unsqueeze(-1) - x_grid
	dist_close = torch.abs(grid_dist) < torch.finfo(torch.double).eps # shape (N_test, N_grid)
	close_pairs = torch.nonzero(dist_close)
	fit[close_pairs[:, 0]] = f_grid[close_pairs[:, 1]]
	noclose = torch.logical_not(dist_close.any(-1))
	div = w_grid / grid_dist[noclose, :]
	fit[noclose] = (div * f_grid).sum(-1) / div.sum(-1)
	diff = result - fit
	print(diff.abs().max().item(), diff.abs().argmax().item(), result[int(diff.abs().argmax().item())].item())
	print((diff[:-1] / result[:-1]).abs().max().item(), (diff[1:] / result[1:]).abs().argmax().item(), result[int((diff[1:] / result[1:]).abs().argmax().item())].item())
	%timeit test_torchquad(r_c, x_test)

eps=5.1448789686149945e-12
2.5 tensor([9.4674e-01, 9.4627e-01, 9.4488e-01, 9.4255e-01, 9.3930e-01, 9.3515e-01,
        9.3009e-01, 9.2415e-01, 9.1734e-01, 9.0969e-01, 9.0122e-01, 8.9194e-01,
        8.8190e-01, 8.7111e-01, 8.5961e-01, 8.4743e-01, 8.3460e-01, 8.2116e-01,
        8.0714e-01, 7.9259e-01, 7.7754e-01, 7.6203e-01, 7.4609e-01, 7.2978e-01,
        7.1312e-01, 6.9617e-01, 6.7895e-01, 6.6151e-01, 6.4389e-01, 6.2613e-01,
        6.0826e-01, 5.9032e-01, 5.7235e-01, 5.5439e-01, 5.3647e-01, 5.1861e-01,
        5.0086e-01, 4.8325e-01, 4.6579e-01, 4.4853e-01, 4.3148e-01, 4.1467e-01,
        3.9812e-01, 3.8186e-01, 3.6590e-01, 3.5026e-01, 3.3495e-01, 3.2000e-01,
        3.0541e-01, 2.9119e-01, 2.7736e-01, 2.6392e-01, 2.5088e-01, 2.3825e-01,
        2.2602e-01, 2.1420e-01, 2.0280e-01, 1.9181e-01, 1.8122e-01, 1.7105e-01,
        1.6128e-01, 1.5191e-01, 1.4294e-01, 1.3436e-01, 1.2616e-01, 1.1834e-01,
        1.1089e-01, 1.0379e-01, 9.7050e-02, 9.0648e-02, 8.4577e-02, 7.8828e-02,
        7

In [8]:
from scipy.integrate import dblquad

def raw_func(x1: float, x2: float, pt1: torch.Tensor, pt2: torch.Tensor, r_c: float, l: torch.Tensor) -> float:
	x: typing.Final[torch.Tensor] = torch.tensor([x1, x2])
	dist1: typing.Final[float] = torch.norm((x - pt1) / l).item()
	dist2: typing.Final[float] = torch.norm((x - pt2) / l).item()
	if dist1 > r_c or dist2 > r_c:
		return 0
	return math.exp(-dist1**2 / 2) * wendland(dist1 / r_c, 1) * math.exp(-dist2**2 / 2) * wendland(dist2 / r_c, 1)

r_c: typing.Final[float] = 2.5
l: typing.Final[torch.Tensor] = torch.ones(2)
pts1: typing.Final[torch.Tensor] = torch.tensor([[0.1, 0], [0, 0], [0.3, 0]]) * l * r_c
for i in range(2):
	for j in range(i + 1, 3):
		dist = torch.norm((pts1[i] - pts1[j]) / (r_c * l)).item()
		print(
			dist,
			dblquad(raw_func, -math.inf, math.inf, -math.inf, math.inf, (pts1[i], pts1[j], r_c, l), eps, eps)[0] / l.prod().item(),
			2 * r_c ** 2 * math.exp(-r_c**2 * dist ** 2 / 4) * dblquad(lambda w, v: math.exp(-r_c**2 * (w**2 + v**2)) * wendland(math.sqrt((w - dist / 2) ** 2 + v ** 2), 1) * wendland(math.sqrt((w + dist / 2) ** 2 + v ** 2), 1), 0, math.sqrt(1 - dist**2 / 4), lambda v: dist / 2 - math.sqrt(1 - v**2), lambda v: math.sqrt(1 - v**2) - dist / 2, (), eps, eps)[0]
		)
pts2: typing.Final[torch.Tensor] = torch.tensor([[0.5, 0], [0, 0], [1.5, 0]]) * l * r_c
for i in range(2):
	for j in range(i + 1, 3):
		dist = torch.norm((pts2[i] - pts2[j]) / (r_c * l)).item()
		print(
			dist,
			dblquad(raw_func, -math.inf, math.inf, -math.inf, math.inf, (pts2[i], pts2[j], r_c, l), eps, eps)[0] / l.prod().item(),
			2 * r_c ** 2 * math.exp(-r_c**2 * dist ** 2 / 4) * dblquad(lambda w, v: math.exp(-r_c**2 * (w**2 + v**2)) * wendland(math.sqrt((w - dist / 2) ** 2 + v ** 2), 1) * wendland(math.sqrt((w + dist / 2) ** 2 + v ** 2), 1), 0, math.sqrt(1 - dist**2 / 4), lambda v: dist / 2 - math.sqrt(1 - v**2), lambda v: math.sqrt(1 - v**2) - dist / 2, (), eps, eps)[0]
		)

0.1 0.9012174662769459 0.9012174662769774
0.2 0.7775397708135832 0.7775397708135036
0.3 0.6082579966440302 0.6082579966439551
0.5 0.2773602034944242 0.2773602034945911
1.0 0.006176941239763823 0.006176941239806725
1.5 2.8957779290611754e-06 2.896022343729426e-06


In [9]:
import numpy as np
import joblib
dists = np.linspace(0., 2., 201)
def dblquad_int(dist: float, r_c: float) -> float:
	return 2 * r_c ** 2 * math.exp(-r_c**2 * dist ** 2 / 4) * dblquad(lambda x, y: math.exp(-r_c**2 * (x**2 + y**2)) * wendland(math.sqrt((x - dist / 2) ** 2 + y ** 2), 1) * wendland(math.sqrt((x + dist / 2) ** 2 + y ** 2), 1), 0, math.sqrt(1 - dist**2 / 4), lambda x: dist / 2 - math.sqrt(1 - x**2), lambda x: math.sqrt(1 - x**2) - dist / 2, (), eps, eps)[0]

jp = joblib.Parallel(n_jobs=-1, verbose=0)
results = torch.tensor(jp(joblib.delayed(lambda dist, r_c: 2 * r_c ** 2 * math.exp(-r_c**2 * dist ** 2 / 4) * dblquad(lambda x, y: math.exp(-r_c**2 * (x**2 + y**2)) * wendland(math.sqrt((x - dist / 2) ** 2 + y ** 2), 1) * wendland(math.sqrt((x + dist / 2) ** 2 + y ** 2), 1), 0, math.sqrt(1 - dist**2 / 4), lambda x: dist / 2 - math.sqrt(1 - x**2), lambda x: math.sqrt(1 - x**2) - dist / 2, (), eps, eps)[0])(dist, r_c) for dist in dists for r_c in np.arange(1, 17) / 2.0)).reshape(dists.size, 16)
result_seq = torch.tensor([dblquad_int(dist, 2.5) for dist in dists])
print((results[:,4] - result_seq).abs().sum().item())
%timeit jp(joblib.delayed(dblquad_int)(dist, r_c) for dist in dists for r_c in np.arange(1, 17) / 2.0)
%timeit jp(joblib.delayed(lambda dist, r_c: 2 * r_c ** 2 * math.exp(-r_c**2 * dist ** 2 / 4) * dblquad(lambda x, y: math.exp(-r_c**2 * (x**2 + y**2)) * wendland(math.sqrt((x - dist / 2) ** 2 + y ** 2), 1) * wendland(math.sqrt((x + dist / 2) ** 2 + y ** 2), 1), 0, math.sqrt(1 - dist**2 / 4), lambda x: dist / 2 - math.sqrt(1 - x**2), lambda x: math.sqrt(1 - x**2) - dist / 2, (), eps, eps)[0])(dist, r_c) for dist in dists for r_c in np.arange(1, 17) / 2.0)

0.0
712 ms ± 41.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
703 ms ± 32.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
